In [0]:
from datetime import datetime
from cdp_dq_framework.utils import notify_email

In [0]:
# Retrieve task values
tv = dbutils.jobs.taskValues
catalog = tv.get(taskKey="derive_config", key="catalog", debugValue=None)
schema_nm = tv.get(taskKey="derive_config", key="schema_nm", debugValue=None)
bucket = tv.get(taskKey="derive_config", key="bucket", debugValue=None)
file_key = tv.get(taskKey="derive_config", key="file_key", debugValue=None)
file_name = tv.get(taskKey="derive_config", key="file_name", debugValue=None)
app_id = tv.get(taskKey="derive_config", key="app_id", debugValue=None)
file_arrival_time = tv.get(taskKey="derive_config", key="file_arrival_time", debugValue=None)
file_arrival_time = datetime.strptime(file_arrival_time, '%Y-%m-%d %H:%M:%S')
app_id = tv.get(taskKey="derive_config", key="app_id", debugValue=None)
excn_id = tv.get(taskKey="derive_config", key="excn_id", debugValue=None)
entity_id = tv.get(taskKey="derive_config", key="entity_id", debugValue=None)
pre_contract_path = tv.get(taskKey="derive_config", key="pre_contract_path", debugValue=None)
pre_contract_file_path = tv.get(taskKey="derive_config", key="pre_contract_file_path", debugValue=None)
post_process_path = tv.get(taskKey="derive_config", key="post_process_path", debugValue=None)
result = tv.get(taskKey="contract_checks", key="result", debugValue=None)
processed_time = datetime.now()
duration = int((processed_time - file_arrival_time).total_seconds())

In [0]:
# Email notification constants
failure_subject = f"CDP validation failed for file {file_name}"
failure_msg = f"Hi Team,\n\nData validation check for the file {file_name} has FAILED, please take a look at the QC item status and make necessary correction.\n\n"
metadata_fail_msg = f"Hi Team,\n\nMeta data defined for the file is Inconsistent with the format of the file {file_name} ,please verity Metadata entries.\n\n Note:\n 1) Keep the filename unique and distinct \n 2) If any contract check flag is set to Y, please ensure the list is not empty \n 3) File delimter defined in Metadata should match to file delimiter \n 4) Set header column to Y or N \n 5) Set email to SNS topic instead of individual email address \n 6) Define date formats in column date_formats or set value to noformat \n 7) Set parquet_header column names matching to databricks table."
greetings = "\n\nRegards,\nCDP"

In [0]:
rejected_path = pre_contract_path.replace("pre_contract", "rejected_files")
rejected_file_path = pre_contract_file_path.replace("pre_contract", "rejected_files")
print("Moving file to rejected path")
dbutils.fs.cp(pre_contract_file_path, rejected_file_path)
print("Removing pre-contract file")
dbutils.fs.rm(pre_contract_file_path)

In [0]:
spark.sql(f"INSERT INTO {catalog}.{schema_nm}.dc_excn_stat VALUES ('{excn_id}', '{app_id}', '{entity_id}', '{file_name}', 'failed', '{file_arrival_time}', '{processed_time}', '{duration}', '{rejected_path}')")
print("Audit table updated with status 'failed'")
failed_checks = result["failed_checks"]
for chk in failed_checks:
    spark.sql(f"INSERT INTO {catalog}.{schema_nm}.dc_error_stat VALUES ('{excn_id}', '{app_id}', '{entity_id}', '{file_name}', '{chk[0]}', '{chk[0]}:\t\t{chk[1]}')")

In [0]:
# Send email notification
email_context = {}
email_context["subject"] = failure_subject
email_context["recipient_list"] = ["udayan.awasthi@takeda.com", "dheeraj.jaiman@takeda.com", "vineet.kumar@takeda.com", "anush.gowda@takeda.com", "astha.chaturvedi@takeda.com"]
if len(result["failed_checks"]) == 1 and failed_checks[0][0] == "Metadata Consistency":
    metadata_fail_msg += greetings
    email_context["message"] = metadata_fail_msg
else:
    for chk in failed_checks:
        failure_msg += f"{chk[0]}:\t{chk[1]}\n"
    failure_msg += greetings
    email_context["message"] = failure_msg
notify_email(email_context)

# raise Exception(f"Contract checks FAILED: {result['failed_checks']}")  # -> job notification
print(f"Contract checks FAILED:\n{result['failed_checks']}")